# 04 — Görev 4: Model Değerlendirme & Açıklanabilirlik

Bu notebook `03_modeling.ipynb`'in kaydettiği modelleri yükler ve şunları üretir:

1. **Detaylı metrik tablosu** — Accuracy / Precision / Recall / F1, üç set için (Train / CV / Test)
2. **ROC analizi** — her model için Train + 5-fold CV-mean (±std) + Test eğrileri, AUC değerleri
3. **Recall–threshold analizi** — karar eşiğine göre recall/precision değişimi
4. **En iyi model + threshold ayarı** — tıbbi vaka için recall öncelikli eşik
5. **SHAP** — `summary_plot` (global) + `force_plot` (tek hasta) → kara kutu açıklaması
6. Streamlit için gerekli artefaktların kaydedilmesi

**Ön koşul:** `01`, `02`, `03` notebook'ları çalıştırılmış olmalı (`models/` ve `data/processed/` dolu olmalı).


In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

warnings.filterwarnings("ignore")

sys.path.append(str(Path.cwd() / "src"))
from src import config as C
from src import utils as U

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")
np.random.seed(C.RANDOM_STATE)
print(f"SHAP {shap.__version__} | random_state={C.RANDOM_STATE}")

c:\Users\sezgi\Masaüstü\odev-yusuf\heart_disease_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP 0.46.0 | random_state=42


## 1. Veri ve modelleri yükle

In [2]:
X_train = pd.read_csv(C.PROC_DIR / "X_train_scaled.csv")
X_test = pd.read_csv(C.PROC_DIR / "X_test_scaled.csv")
y_train = pd.read_csv(C.PROC_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(C.PROC_DIR / "y_test.csv").squeeze("columns")

# Scale EDİLMEMİŞ test verisi — SHAP grafiklerinde gerçek değerleri göstermek için
X_test_display = pd.read_csv(C.PROC_DIR / "X_test.csv")

selected = joblib.load(C.MODELS_DIR / "selected_features.joblib")
X_train = X_train[selected]
X_test = X_test[selected]
X_test_display = X_test_display[selected]

model_names = ["LogisticRegression", "RandomForest", "XGBoost"]
models = {n: joblib.load(C.MODELS_DIR / f"{n}.joblib") for n in model_names}
best_name = joblib.load(C.MODELS_DIR / "best_model_name.joblib")

print(f"Yüklenen modeller: {model_names}")
print(f"En iyi model (Görev 3'ten): {best_name}")
print(f"Train: {X_train.shape} | Test: {X_test.shape} | Özellik: {len(selected)}")

Yüklenen modeller: ['LogisticRegression', 'RandomForest', 'XGBoost']
En iyi model (Görev 3'ten): LogisticRegression
Train: (227, 7) | Test: (61, 7) | Özellik: 7


## 2. Detaylı Metrik Tablosu (Train / CV / Test)

- **Train:** model tüm eğitim setinde fit edilip yine eğitim setinde değerlendirilir.
- **CV:** 5-fold StratifiedKFold çapraz doğrulama (mean ± std) — gerçek "doğrulama" performansı.
- **Test:** dokunulmamış test seti — gerçek genelleme tahmini.

Tıbbi vaka olduğu için tabloyu **test recall**'a göre sıralıyoruz.

In [3]:
cv = StratifiedKFold(n_splits=C.CV_FOLDS, shuffle=True, random_state=C.RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1"]

def set_metrics(model, X, y):
    pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
    }

rows = []
for name, model in models.items():
    train_m = set_metrics(model, X_train, y_train)
    test_m = set_metrics(model, X_test, y_test)
    cv_res = cross_validate(clone(model), X_train, y_train, cv=cv,
                            scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for m in scoring:
        row[f"train_{m}"] = round(train_m[m], 4)
        row[f"cv_{m}_mean"] = round(cv_res[f"test_{m}"].mean(), 4)
        row[f"cv_{m}_std"] = round(cv_res[f"test_{m}"].std(), 4)
        row[f"test_{m}"] = round(test_m[m], 4)
    rows.append(row)

metrics_df = pd.DataFrame(rows).sort_values("test_recall", ascending=False).reset_index(drop=True)
U.save_table(metrics_df, C.METRICS_DIR / "evaluation_metrics")
print("Kaydedildi → outputs/metrics/evaluation_metrics.{csv,md}\n")
metrics_df

Kaydedildi → outputs/metrics/evaluation_metrics.{csv,md}



,model,train_accuracy,cv_accuracy_mean,cv_accuracy_std,test_accuracy,train_precision,cv_precision_mean,cv_precision_std,test_precision,train_recall,cv_recall_mean,cv_recall_std,test_recall,train_f1,cv_f1_mean,cv_f1_std,test_f1
0,LogisticRegression,0.7885,0.7882,0.0528,0.8689,0.7547,0.7473,0.0477,0.8125,0.7843,0.7933,0.1070,0.9286,0.7692,0.7676,0.0716,0.8667
1,RandomForest,0.9295,0.7706,0.0609,0.8689,0.9388,0.7356,0.0402,0.8333,0.9020,0.7533,0.1301,0.8929,0.9200,0.7414,0.0846,0.8621
2,XGBoost,0.9956,0.7398,0.0521,0.8033,0.9903,0.7147,0.0702,0.7857,1.0000,0.7052,0.0727,0.7857,0.9951,0.7083,0.0617,0.7857


In [4]:
# Görsel — 3 model için recall'in Train / CV / Test karşılaştırması
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics_df))
w = 0.25
ax.bar(x - w, metrics_df["train_recall"], w, label="Train", color="#5B8DEF")
ax.bar(x, metrics_df["cv_recall_mean"], w, yerr=metrics_df["cv_recall_std"],
       capsize=4, label="CV (±std)", color="#F0AD4E")
ax.bar(x + w, metrics_df["test_recall"], w, label="Test", color="#D9534F")
ax.set_xticks(x)
ax.set_xticklabels(metrics_df["model"])
ax.set_ylabel("Recall")
ax.set_ylim(0, 1.05)
ax.set_title("Recall karşılaştırması — Train / CV / Test")
ax.legend()
U.savefig(fig, C.FIG_DIR / "08_recall_train_cv_test.png")
print("Kaydedildi → outputs/figures/08_recall_train_cv_test.png")

Kaydedildi → outputs/figures/08_recall_train_cv_test.png


## 3. ROC Analizi — Train + CV-mean + Test

Her model için tek figürde üç eğri:
- **Train ROC** — tüm eğitim setinde
- **CV-mean ROC** — 5 fold'un ortalaması, gri bant ±1 standart sapma (doğrulama performansı)
- **Test ROC** — test setinde

**Overfitting yorumu:** Train AUC ile CV/Test AUC arasındaki fark büyükse model
ezberlemiş demektir. Train ≈ CV ≈ Test ise model sağlıklı genelliyor.

In [5]:
def cv_roc_curve(model, X, y, cv):
    """5-fold CV out-of-fold ROC: ortak FPR gridinde ortalama TPR ve std."""
    mean_fpr = np.linspace(0, 1, 100)
    tprs, aucs = [], []
    for tr_idx, val_idx in cv.split(X, y):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba = m.predict_proba(X.iloc[val_idx])[:, 1]
        fpr, tpr, _ = roc_curve(y.iloc[val_idx], proba)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(auc(fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    return mean_fpr, mean_tpr, np.std(tprs, axis=0), np.mean(aucs), np.std(aucs)


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
roc_summary = []

for ax, (name, model) in zip(axes, models.items()):
    # Train ROC
    proba_tr = model.predict_proba(X_train)[:, 1]
    fpr_tr, tpr_tr, _ = roc_curve(y_train, proba_tr)
    auc_tr = auc(fpr_tr, tpr_tr)
    # Test ROC
    proba_te = model.predict_proba(X_test)[:, 1]
    fpr_te, tpr_te, _ = roc_curve(y_test, proba_te)
    auc_te = auc(fpr_te, tpr_te)
    # CV-mean ROC
    m_fpr, m_tpr, s_tpr, auc_cv, auc_cv_std = cv_roc_curve(model, X_train, y_train, cv)

    ax.plot(fpr_tr, tpr_tr, color="#5B8DEF", lw=2,
            label=f"Train (AUC={auc_tr:.3f})")
    ax.plot(m_fpr, m_tpr, color="#F0AD4E", lw=2,
            label=f"CV-mean (AUC={auc_cv:.3f}±{auc_cv_std:.3f})")
    ax.fill_between(m_fpr, np.maximum(m_tpr - s_tpr, 0),
                    np.minimum(m_tpr + s_tpr, 1), color="#F0AD4E", alpha=0.2)
    ax.plot(fpr_te, tpr_te, color="#D9534F", lw=2,
            label=f"Test (AUC={auc_te:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC — {name}")
    ax.legend(loc="lower right", fontsize=9)

    roc_summary.append({"model": name, "train_auc": round(auc_tr, 4),
                        "cv_auc_mean": round(auc_cv, 4), "cv_auc_std": round(auc_cv_std, 4),
                        "test_auc": round(auc_te, 4),
                        "train_minus_cv": round(auc_tr - auc_cv, 4)})

fig.suptitle("ROC Eğrileri — Train / CV-mean / Test", y=1.03, fontsize=14)
U.savefig(fig, C.FIG_DIR / "09_roc_curves.png")

roc_df = pd.DataFrame(roc_summary)
U.save_table(roc_df, C.METRICS_DIR / "roc_auc_summary")
print("Kaydedildi → outputs/figures/09_roc_curves.png")
print("Kaydedildi → outputs/metrics/roc_auc_summary.{csv,md}\n")
roc_df

Kaydedildi → outputs/figures/09_roc_curves.png
Kaydedildi → outputs/metrics/roc_auc_summary.{csv,md}



,model,train_auc,cv_auc_mean,cv_auc_std,test_auc,train_minus_cv
0,LogisticRegression,0.8900,0.8769,0.0540,0.9502,0.0131
1,RandomForest,0.9880,0.8413,0.0668,0.9340,0.1467
2,XGBoost,0.9998,0.8077,0.0621,0.9058,0.1921


**Tablodan okunacaklar (rapora):**
`train_minus_cv` sütunu küçükse (≈ 0–0.05) model sağlıklı genelliyor demektir.
Ağaç tabanlı modellerde (RF, XGBoost) train AUC genelde 1.0'a yakın çıkar; asıl
bakılması gereken CV ve Test AUC'nin birbirine ve train'e ne kadar yakın olduğudur.

## 4. Recall — Threshold Analizi

Modeller varsayılan olarak 0.5 olasılık eşiğiyle karar verir. Tıbbi vakada **hasta
kaçırmanın (false negative) maliyeti yüksek** olduğu için eşiği düşürerek recall'ı
artırabiliriz — bunun bedeli precision'ın düşmesidir. Aşağıdaki grafik bu ödünleşmeyi
test seti üzerinde gösterir.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    proba = model.predict_proba(X_test)[:, 1]
    prec, rec, thr = precision_recall_curve(y_test, proba)
    # precision_recall_curve: thr uzunlugu prec/rec'ten 1 eksik
    ax.plot(thr, rec[:-1], color="#D9534F", lw=2, label="Recall")
    ax.plot(thr, prec[:-1], color="#5B8DEF", lw=2, label="Precision")
    ax.axvline(0.5, color="gray", ls="--", lw=1, label="Varsayılan eşik (0.5)")
    ax.set_xlabel("Karar eşiği (threshold)")
    ax.set_ylabel("Skor")
    ax.set_title(f"Recall / Precision vs Threshold — {name}")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
fig.suptitle("Recall–Threshold Analizi (test seti)", y=1.03, fontsize=14)
U.savefig(fig, C.FIG_DIR / "10_recall_threshold.png")
print("Kaydedildi → outputs/figures/10_recall_threshold.png")

Kaydedildi → outputs/figures/10_recall_threshold.png


## 5. En İyi Model + Threshold Ayarı

En iyi model üzerinde optimal eşiği **Youden's J istatistiği** (J = TPR − FPR, ROC
eğrisinde sol-üst köşeye en yakın nokta) ile belirliyoruz. Bu, recall ve precision
arasında dengeli ama recall lehine bir nokta verir. Varsayılan 0.5 ile karşılaştırıyoruz.

In [7]:
best_model = models[best_name]
proba_test = best_model.predict_proba(X_test)[:, 1]

# Youden's J ile optimal esik
fpr, tpr, thr = roc_curve(y_test, proba_test)
youden_j = tpr - fpr
best_thr = float(thr[np.argmax(youden_j)])
print(f"En iyi model: {best_name}")
print(f"Youden's J optimal eşik: {best_thr:.3f}  (varsayılan: 0.500)\n")

# Iki esikte confusion matrix + metrikler
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (t, label) in zip(axes, [(0.5, "Varsayılan (0.5)"),
                                  (best_thr, f"Ayarlı ({best_thr:.3f})")]):
    pred = (proba_test >= t).astype(int)
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Sağlıklı", "Hasta"]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    rec = recall_score(y_test, pred, zero_division=0)
    prec = precision_score(y_test, pred, zero_division=0)
    ax.set_title(f"{label}\nRecall={rec:.3f} | Precision={prec:.3f}")
fig.suptitle(f"{best_name} — Eşik karşılaştırması (test)", y=1.05, fontsize=13)
U.savefig(fig, C.FIG_DIR / "11_threshold_confusion.png")
print("Kaydedildi → outputs/figures/11_threshold_confusion.png")

En iyi model: LogisticRegression
Youden's J optimal eşik: 0.452  (varsayılan: 0.500)

Kaydedildi → outputs/figures/11_threshold_confusion.png


## 6. SHAP — Kara Kutu Açıklaması

SHAP (SHapley Additive exPlanations), her özelliğin bir tahmine ne kadar ve hangi
yönde katkı yaptığını oyun teorisindeki Shapley değerleriyle hesaplar.
- Ağaç tabanlı modeller (RF / XGBoost) → `TreeExplainer` (hızlı, kesin)
- Lineer modeller (LogReg) → `LinearExplainer`

In [9]:
# Arka plan örneği (LinearExplainer için gerekli; ayrıca Streamlit'e kaydedilecek)
bg_n = min(100, len(X_train))
background = X_train.sample(n=bg_n, random_state=C.RANDOM_STATE)

if best_name in ("RandomForest", "XGBoost"):
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.LinearExplainer(best_model, background)

shap_exp = explainer(X_test)

# İkili sınıflandırmada bazı explainer'lar 3B döner -> pozitif sınıfı (hasta) al
if shap_exp.values.ndim == 3:
    shap_exp = shap_exp[:, :, 1]

# SHAP değerleri scale'li veriyle hesaplandı (doğru) ama grafiklerde gerçek
# (scale edilmemiş) değerleri göstermek okunabilirliği artırır: chol=243 mg/dl
# gibi -> "chol=1.45" z-skoru yerine. .data alanını gerçek değerlerle değiştir.
shap_exp.data = X_test_display.values

print(f"Explainer: {type(explainer).__name__}")
print(f"SHAP değerleri şekli: {shap_exp.values.shape}")

Explainer: LinearExplainer
SHAP değerleri şekli: (61, 7)


### 6a. Summary plot — tüm özelliklerin global etkisi

In [10]:
fig = plt.figure()
shap.summary_plot(shap_exp, X_test_display, show=False)
U.savefig(plt.gcf(), C.FIG_DIR / "12_shap_summary.png")
print("Kaydedildi → outputs/figures/12_shap_summary.png")

Kaydedildi → outputs/figures/12_shap_summary.png


**Yorum (rapora):** Her satır bir özellik; noktalar hastalar. Sağa (pozitif SHAP)
giden nokta tahmini "hasta" yönünde, sola giden "sağlıklı" yönünde iter. Renk
özelliğin değerini gösterir (kırmızı=yüksek, mavi=düşük). En üstteki özellikler
modelin kararında en etkili olanlardır.

### 6b. Force plot — tek bir hastanın açıklaması

In [11]:
# Açıklanacak örnek hastayı seç (ilk test hastası)
patient_idx = 0
proba_p = best_model.predict_proba(X_test.iloc[[patient_idx]])[:, 1][0]
print(f"Hasta #{patient_idx} — model tahmini (hasta olma olasılığı): {proba_p:.1%}")
print(f"Gerçek etiket: {'Hasta' if y_test.iloc[patient_idx]==1 else 'Sağlıklı'}\n")

shap.force_plot(
    base_value=shap_exp[patient_idx].base_values,
    shap_values=shap_exp[patient_idx].values,
    features=X_test_display.iloc[patient_idx],   # gerçek (scale edilmemiş) değerler
    matplotlib=True,
    show=False,
)
U.savefig(plt.gcf(), C.FIG_DIR / "13_shap_force_patient.png")
print("Kaydedildi → outputs/figures/13_shap_force_patient.png")

Hasta #0 — model tahmini (hasta olma olasılığı): 30.1%
Gerçek etiket: Sağlıklı

Kaydedildi → outputs/figures/13_shap_force_patient.png


**Yorum (rapora):** Kırmızı oklar tahmini yukarı (hasta yönünde), mavi oklar
aşağı (sağlıklı yönünde) iten özelliklerdir. Ok uzunluğu katkının büyüklüğüdür.
`base value` ortalama tahmin, `f(x)` bu hasta için modelin çıktısıdır. Bu grafik
"model neden bu kararı verdi" sorusunu somut olarak cevaplar.

### 6c. Waterfall plot (bonus — aynı hasta, daha okunaklı)

In [12]:
fig = plt.figure()
shap.plots.waterfall(shap_exp[patient_idx], show=False)
U.savefig(plt.gcf(), C.FIG_DIR / "14_shap_waterfall_patient.png")
print("Kaydedildi → outputs/figures/14_shap_waterfall_patient.png")

Kaydedildi → outputs/figures/14_shap_waterfall_patient.png


## 7. Streamlit için artefaktları kaydet

`05` adımındaki Streamlit uygulaması bu dosyaları kullanacak:
- En iyi model zaten `models/{best_name}.joblib` olarak kayıtlı
- Karar eşiği, SHAP arka plan örneği ve özellik listesi burada kaydedilir

In [13]:
joblib.dump(best_thr, C.MODELS_DIR / "decision_threshold.joblib")
background.to_csv(C.MODELS_DIR / "shap_background.csv", index=False)

deploy_info = {
    "best_model_file": f"{best_name}.joblib",
    "best_model_name": best_name,
    "decision_threshold": round(best_thr, 4),
    "selected_features": selected,
    "continuous_features": [f for f in C.CONTINUOUS if f in selected],
}
joblib.dump(deploy_info, C.MODELS_DIR / "deploy_info.joblib")

print("Streamlit için kaydedildi:")
print(f"  models/decision_threshold.joblib  (eşik = {best_thr:.3f})")
print(f"  models/shap_background.csv        ({bg_n} satır)")
print(f"  models/deploy_info.joblib")

Streamlit için kaydedildi:
  models/decision_threshold.joblib  (eşik = 0.452)
  models/shap_background.csv        (100 satır)
  models/deploy_info.joblib


## Üretilen çıktılar:


- `outputs/figures/08_recall_train_cv_test.png`
- `outputs/figures/09_roc_curves.png`
- `outputs/figures/10_recall_threshold.png`
- `outputs/figures/11_threshold_confusion.png`
- `outputs/figures/12_shap_summary.png`
- `outputs/figures/13_shap_force_patient.png`
- `outputs/figures/14_shap_waterfall_patient.png`
- `outputs/metrics/evaluation_metrics.{csv,md}`
- `outputs/metrics/roc_auc_summary.{csv,md}`
- `models/decision_threshold.joblib`, `shap_background.csv`, `deploy_info.joblib`


**Rapor için hazır malzeme:** `outputs/metrics/` altındaki `.md` tablolarını doğrudan
PPT/Word'e yapıştırabilir, `outputs/figures/` altındaki grafikleri slaytlarda kullanılacak